In [1]:
from pathlib import Path
import os
import time
import random
import torch

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_transformer import SingleOutTransformerNet
from metrics import eval_single_metrics_test

In [2]:
OUTDIR = Path("./Results_TR")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path("./trained_models")
MODELSDIR.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


#### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

##### Dataframes

In [7]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s, y_test_s, BATCH_SIZE)

#### Models

### Training

In [8]:
EPOCHS = 150
# METRICS_EVERY = int(EPOCHS/8)
METRICS_EVERY = 1
LR = 1e-3
WD = 1e-5

In [9]:
IN_DIM = X_train_s.shape[1]

EMB_DIM = 64
NHEAD = 4
# NUM_LAYERS = 3
NUM_LAYERS = 5
FF_DIM = 128
DROPOUT = 0.1

In [10]:
tag = "single_TR_age"

torch.manual_seed(SEED)    
model_single = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                       num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                       dropout=DROPOUT).to(DEVICE)    

# loss = torch.nn.MSELoss()
loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(f"Results/train_history_{NUM_LAYERS}layers.csv", index=False)
torch.save(model_single.state_dict(), MODELSDIR / f"TR_model_{NUM_LAYERS}layers.pt")

ep=001 tr_loss=0.3085 | va_loss=0.2123 | tr R2=0.563 | va R2=0.529 | tr RMSE=0.661 | va RMSE=0.671 | tr MAE=0.520 | va MAE=0.530 | lr=1.00e-03 | bad_epochs=0
ep=002 tr_loss=0.1887 | va_loss=0.1814 | tr R2=0.640 | va R2=0.600 | tr RMSE=0.600 | va RMSE=0.618 | tr MAE=0.471 | va MAE=0.485 | lr=1.00e-03 | bad_epochs=0
ep=003 tr_loss=0.1790 | va_loss=0.1819 | tr R2=0.640 | va R2=0.602 | tr RMSE=0.600 | va RMSE=0.617 | tr MAE=0.474 | va MAE=0.486 | lr=1.00e-03 | bad_epochs=1
ep=004 tr_loss=0.1712 | va_loss=0.1712 | tr R2=0.666 | va R2=0.625 | tr RMSE=0.578 | va RMSE=0.598 | tr MAE=0.450 | va MAE=0.468 | lr=1.00e-03 | bad_epochs=0
ep=005 tr_loss=0.1658 | va_loss=0.1705 | tr R2=0.668 | va R2=0.630 | tr RMSE=0.576 | va RMSE=0.595 | tr MAE=0.453 | va MAE=0.472 | lr=1.00e-03 | bad_epochs=0
ep=006 tr_loss=0.1641 | va_loss=0.1756 | tr R2=0.664 | va R2=0.616 | tr RMSE=0.580 | va RMSE=0.606 | tr MAE=0.453 | va MAE=0.474 | lr=1.00e-03 | bad_epochs=1
ep=007 tr_loss=0.1589 | va_loss=0.2022 | tr R2=0.606

In [11]:
m_te = eval_single_metrics_test(model_single, X_test_s, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [12]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)
single_metrics_path = f"Results/test_metrics_{NUM_LAYERS}layers.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [13]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.69801,7.49474,5.841167
